# 09. Traffic-Sign Detector (YOLO) for Multi-Sign Street Photos

The classifier from notebooks 04–05 only works on a **tight crop of one sign**: on whole street photos it is almost always wrong, because a sign covers well under 1% of a 4K photo and shrinks to a few pixels at 224 × 224.

This notebook adds the missing first stage of a **two-stage pipeline**:

1. **Detector (new, trained here):** a 1-class YOLO model that finds every traffic sign in a street photo.
2. **Classifier (unchanged):** the final fine-tuned ResNet-18 classifies each detected box into one of the 26 classes.

**What does NOT change:** the ResNet-18 checkpoint, its 98.70% test result, the dataset, and the split. The detector uses the **same train / val / test split by source photo**, so it never sees a test photo during training.

**Labels:** the raw dataset already ships YOLO-format boxes (`dataset/raw/labels/*.txt`, one class `0 = Traffic Sign`), the format used by the dataset author's own YOLOv8 detector.

**Runtime:** GPU required (**Runtime → Change runtime type → T4 GPU**). Training takes roughly 1–2 hours. Outputs are saved to Google Drive, so a disconnected run can be resumed (see section 4).

## 1. Setup: Drive, repository, paths

In [ ]:
import json
import random
import shutil
import subprocess
import sys
import time
from pathlib import Path

# ---- Edit these if your setup differs ------------------------------------
REPO_URL = "https://github.com/RothRothRoth/DeepLearning_CNN.git"
REPO_BRANCH = "main"
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/Road_Sign_Recognition_CNN")
YOLO_WEIGHTS = "yolov8s.pt"   # COCO-pretrained starting point (downloaded by Ultralytics)
IMGSZ = 1280                  # signs are tiny: 640 would shrink most of them below 10 px
EPOCHS = 50
BATCH = 8
SEED = 42
# --------------------------------------------------------------------------

from google.colab import drive

drive.mount("/content/drive")
PROJECT_ROOT = Path("/content/DeepLearning_CNN")
if (PROJECT_ROOT / ".git").exists():
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)
%cd {PROJECT_ROOT}
sys.path.insert(0, str(PROJECT_ROOT))

RAW_IMAGES = DRIVE_PROJECT_DIR / "dataset" / "raw" / "images"
RAW_LABELS = DRIVE_PROJECT_DIR / "dataset" / "raw" / "labels"
SPLITS_DIR = PROJECT_ROOT / "dataset" / "splits"          # committed split manifests
CLASSIFIER_CKPT = DRIVE_PROJECT_DIR / ("results/runs/hyperparameter_tuning/resnet18_finetuned_lr_wd_seed42/"
                                       "final_tuned_model/resnet18_finetuned_tuned_best.pt")
OUTPUT_DIR = DRIVE_PROJECT_DIR / "results" / "runs" / "sign_detector" / f"yolov8s_{IMGSZ}_seed{SEED}"
YOLO_DATA = Path("/content/yolo_signs")                    # fast local copy used for training

for label, path in {"raw street photos": RAW_IMAGES, "raw YOLO labels": RAW_LABELS,
                    "split manifests": SPLITS_DIR, "final classifier checkpoint": CLASSIFIER_CKPT}.items():
    print(f"{'OK     ' if path.exists() else 'MISSING'}  {label}: {path}")
missing = [p for p in (RAW_IMAGES, RAW_LABELS, SPLITS_DIR, CLASSIFIER_CKPT) if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing: {missing}. Check the Drive folder layout (see notebook 01).")

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: switch the runtime to GPU")
assert torch.cuda.is_available(), "A GPU runtime is required for training."

## 2. Install Ultralytics (Colab only)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3,<9"], check=True)
import ultralytics
from ultralytics import YOLO

ultralytics.checks()
ULTRALYTICS_VERSION = ultralytics.__version__

## 3. Build the YOLO dataset with the existing leak-free split

Every street photo goes to the **same split as in the classifier experiments** (from `dataset/splits/*.csv`, column `parent_filename`). Photos that contain no sign of the 26 classes (only `UNDEFINED` or other signs) are not in the manifests; they go to **train only**, so the test photos stay exactly the classifier's test photos.

Images are resized to 1920 px wide for faster loading; YOLO labels are normalised (0–1), so they stay valid.

In [ ]:
import pandas as pd
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True   # a few Drive copies end a couple of bytes early
unreadable = []                          # (file, size in bytes, error) for files that never opened


def load_photo(path, tries=3):
    for attempt in range(tries):
        try:
            return Image.open(path).convert("RGB")
        except Exception as e:  # damaged / incompletely synced Drive copy, or a transient Drive read error
            err = f"{type(e).__name__}: {e}"
            time.sleep(2)
    unreadable.append((path.name, path.stat().st_size, err))
    return None


splits = {s: pd.read_csv(SPLITS_DIR / f"{s}.csv") for s in ("train", "val", "test")}
parent_split = {}
for s, df in splits.items():
    for p in df["parent_filename"].unique():
        assert parent_split.setdefault(p, s) == s, f"{p} appears in two splits"
print("Source photos per split:", {s: df["parent_filename"].nunique() for s, df in splits.items()})

all_images = sorted(p.name for p in RAW_IMAGES.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
print(f"Raw photos on Drive: {len(all_images)} | not in the manifests (-> train only): "
      f"{sum(n not in parent_split for n in all_images)}")

if YOLO_DATA.exists():
    shutil.rmtree(YOLO_DATA)
stats = {s: {"images": 0, "boxes": 0, "skipped_unreadable": 0} for s in ("train", "val", "test")}
t0 = time.time()
for i, name in enumerate(all_images, 1):
    split = parent_split.get(name, "train")
    img = load_photo(RAW_IMAGES / name)
    if img is None:
        stats[split]["skipped_unreadable"] += 1
        continue
    (YOLO_DATA / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DATA / "labels" / split).mkdir(parents=True, exist_ok=True)
    if img.width > 1920:
        img = img.resize((1920, round(img.height * 1920 / img.width)))
    img.save(YOLO_DATA / "images" / split / name, quality=95)
    label = RAW_LABELS / (Path(name).stem + ".txt")
    if label.is_file():
        shutil.copy(label, YOLO_DATA / "labels" / split / label.name)
        stats[split]["boxes"] += sum(1 for line in label.read_text().splitlines() if line.strip())
    stats[split]["images"] += 1
    if i % 500 == 0:
        print(f"  {i}/{len(all_images)} photos, {time.time() - t0:.0f}s")

print(f"Built in {time.time() - t0:.0f}s:", json.dumps(stats))
print(f"Unreadable files on Drive: {len(unreadable)}")
for u in unreadable:
    print("  ", u)

bad_eval = [u[0] for u in unreadable if parent_split.get(u[0], "train") != "train"]
if bad_eval:
    raise RuntimeError(f"Val/test photos unreadable on Drive: {bad_eval}. Replace them on Drive with the "
                       "laptop copies (dataset/raw/images), then re-run this cell.")
assert {p for p, s in parent_split.items() if s == "test"} == {p.name for p in (YOLO_DATA / "images" / "test").iterdir()}
print("Test photos are exactly the classifier's test photos.")

DATA_YAML = YOLO_DATA / "data.yaml"
DATA_YAML.write_text(f"path: {YOLO_DATA}\ntrain: images/train\nval: images/val\ntest: images/test\n"
                     "names:\n  0: Traffic Sign\n")

### Sanity check: draw the ground-truth boxes on three training photos

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

random.seed(SEED)
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, path in zip(axes, random.sample(sorted((YOLO_DATA / "images" / "train").iterdir()), 3)):
    img = Image.open(path)
    ax.imshow(img)
    label = YOLO_DATA / "labels" / "train" / (path.stem + ".txt")
    for line in (label.read_text().splitlines() if label.exists() else []):
        _, xc, yc, w, h = map(float, line.split())
        ax.add_patch(patches.Rectangle(((xc - w / 2) * img.width, (yc - h / 2) * img.height),
                                       w * img.width, h * img.height, fill=False, color="cyan", lw=2))
    ax.set_title(path.name[:40], fontsize=9)
    ax.axis("off")
plt.show()

## 4. Train the detector

Checkpoints are written to Google Drive after every epoch. **If Colab disconnects**, re-run sections 1–3, set `RESUME = True` below and run this cell again: training continues from the last saved epoch.

In [ ]:
RESUME = False
TRAIN_DIR = OUTPUT_DIR / "train"
last = TRAIN_DIR / "weights" / "last.pt"

t0 = time.time()
if RESUME and last.exists():
    YOLO(str(last)).train(resume=True)
else:
    if TRAIN_DIR.exists() and not RESUME:
        raise FileExistsError(f"{TRAIN_DIR} already exists. Set RESUME = True, or rename/delete it on Drive yourself.")
    YOLO(YOLO_WEIGHTS).train(data=str(DATA_YAML), imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, seed=SEED,
                             deterministic=True, patience=15, workers=2,
                             project=str(OUTPUT_DIR), name="train", exist_ok=False)
TRAIN_MINUTES = (time.time() - t0) / 60
BEST = TRAIN_DIR / "weights" / "best.pt"
print(f"Training time this session: {TRAIN_MINUTES:.1f} min | best weights: {BEST} ({BEST.stat().st_size / 1e6:.1f} MB)")

## 5. Detector results on the held-out test photos

Standard detection metrics for the **test split only** (evaluated once, after training):
- **Precision:** of the boxes the detector draws, how many are real signs.
- **Recall:** of the real signs, how many are found.
- **mAP@50 / mAP@50-95:** average precision at IoU 0.5 / averaged over IoU 0.5 to 0.95.

In [ ]:
detector = YOLO(str(BEST))
test_metrics = detector.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH,
                            project=str(OUTPUT_DIR), name="test_eval", exist_ok=True)
det = {"precision": float(test_metrics.box.mp), "recall": float(test_metrics.box.mr),
       "mAP50": float(test_metrics.box.map50), "mAP50_95": float(test_metrics.box.map)}
print(json.dumps(det, indent=2))

## 6. End-to-end pipeline on the test photos: detect → crop → classify

For every **test** photo, the detector finds boxes; each box is cropped from the **original full-resolution photo** and classified by the **unchanged** final ResNet-18. Each ground-truth sign of the 26 classes (from `test.csv`) is then matched to the detection with the highest IoU (≥ 0.5).

- **Detection recall (26 classes):** share of the 463 test signs that were found.
- **End-to-end accuracy:** share of the 463 test signs that were found **and** correctly classified. This is the realistic number for full street photos; compare it with the classifier's 98.70% on perfect crops.
- **Unmatched detections:** boxes that match no 26-class sign. They are false positives **or** real signs outside the 26 classes (e.g. `UNDEFINED`, STOP), which the classifier cannot name.

In [ ]:
from src.config import CLASS_NAMES, IMAGE_SIZE, NUM_CLASSES
from src.data.transforms import get_eval_transforms
from src.models import create_resnet18_finetuned
from src.training import load_checkpoint

device = "cuda"
classifier = create_resnet18_finetuned(num_classes=NUM_CLASSES)
load_checkpoint(CLASSIFIER_CKPT, model=classifier, map_location=device)
classifier.to(device).eval()
tf = get_eval_transforms(IMAGE_SIZE)
CONF = 0.25


def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-9)


test_df = splits["test"]
rows, unmatched = [], 0
for parent, gt in test_df.groupby("parent_filename"):
    photo = Image.open(RAW_IMAGES / parent).convert("RGB")
    boxes = detector.predict(photo, imgsz=IMGSZ, conf=CONF, verbose=False)[0].boxes
    dets = [(list(map(float, b)), float(c)) for b, c in zip(boxes.xyxy.cpu(), boxes.conf.cpu())]
    preds = []
    with torch.no_grad():
        for (x1, y1, x2, y2), _ in dets:
            crop = photo.crop((int(x1), int(y1), int(x2), int(y2)))
            p = torch.softmax(classifier(tf(crop).unsqueeze(0).to(device)), 1)[0].cpu()
            preds.append((CLASS_NAMES[int(p.argmax())], float(p.max())))
    used = set()
    for _, g in gt.iterrows():
        gbox = (g["bbox_x"], g["bbox_y"], g["bbox_x"] + g["bbox_w"], g["bbox_y"] + g["bbox_h"])
        best, best_iou = None, 0.5
        for i, (dbox, _) in enumerate(dets):
            v = iou(gbox, dbox)
            if i not in used and v >= best_iou:
                best, best_iou = i, v
        if best is not None:
            used.add(best)
        name, conf = preds[best] if best is not None else (None, None)
        rows.append({"parent": parent, "crop_path": g["crop_path"], "true": g["class_name"],
                     "detected": best is not None, "iou": best_iou if best is not None else 0.0,
                     "pred": name, "conf": conf, "correct": name == g["class_name"]})
    unmatched += len(dets) - len(used)

e2e = pd.DataFrame(rows)
summary = {
    "test_signs_26_classes": len(e2e),
    "detection_recall_26_classes": float(e2e["detected"].mean()),
    "end_to_end_accuracy": float(e2e["correct"].mean()),
    "classifier_accuracy_on_detected_signs": float(e2e.loc[e2e["detected"], "correct"].mean()),
    "unmatched_detections": int(unmatched),
    "detector_conf_threshold": CONF,
}
print(json.dumps(summary, indent=2))
print("\nMissed or misclassified, by class:")
print(e2e[~e2e["correct"]].groupby("true").size().sort_values(ascending=False).to_string())

## 7. Save results to Drive

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
e2e.to_csv(OUTPUT_DIR / "end_to_end_test_predictions.csv", index=False)
report = {
    "detector": {"weights": YOLO_WEIGHTS, "imgsz": IMGSZ, "epochs_requested": EPOCHS, "batch": BATCH, "seed": SEED,
                 "ultralytics_version": ULTRALYTICS_VERSION, "best_weights": str(BEST),
                 "train_minutes_last_session": round(TRAIN_MINUTES, 1) if "TRAIN_MINUTES" in globals() else None},
    "dataset": {"split_source": "dataset/splits/*.csv (same split as the classifier)", "build_stats": stats},
    "detector_test_metrics": det,
    "end_to_end_test": summary,
    "classifier_checkpoint": str(CLASSIFIER_CKPT),
}
(OUTPUT_DIR / "detector_and_pipeline_metrics.json").write_text(json.dumps(report, indent=2))
print("Saved to", OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.iterdir()):
    print("  ", p.name)

## 8. Visual check: the full pipeline on four test photos

In [ ]:
from PIL import ImageDraw

sample = random.Random(SEED).sample(sorted(test_df["parent_filename"].unique()), 4)
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
for ax, parent in zip(axes.ravel(), sample):
    photo = Image.open(RAW_IMAGES / parent).convert("RGB")
    draw = ImageDraw.Draw(photo)
    for b in detector.predict(photo, imgsz=IMGSZ, conf=CONF, verbose=False)[0].boxes.xyxy.cpu():
        x1, y1, x2, y2 = map(int, b)
        with torch.no_grad():
            p = torch.softmax(classifier(tf(photo.crop((x1, y1, x2, y2))).unsqueeze(0).to(device)), 1)[0].cpu()
        draw.rectangle([x1, y1, x2, y2], outline=(56, 189, 248), width=8)
        draw.text((x1, max(0, y1 - 40)), f"{CLASS_NAMES[int(p.argmax())]} {100 * float(p.max()):.0f}%",
                  fill=(255, 255, 0))
    ax.imshow(photo)
    ax.set_title(parent, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9. Next steps

1. **Download the detector weights** from Drive:
   `MyDrive/Road_Sign_Recognition_CNN/results/runs/sign_detector/yolov8s_1280_seed42/train/weights/best.pt`
   and save them on the laptop as `D:\DL_checkpoint\sign_detector_best.pt` (about 20 MB).
2. Report the numbers from sections 5 and 6 (detector precision / recall / mAP, detection recall and end-to-end accuracy). They are **new results**, separate from the classifier's 98.70%.
3. The web app can then add a "detect all signs in a street photo" mode that uses this detector plus the unchanged classifier.

**Caveats to state in the report:** single run (seed 42); the detector finds *all* traffic signs, including signs outside the 26 classes, which the classifier will still force into one of its 26 classes; some test photos are video frames close in time to training frames (see the leakage check), which may make the test photos easier than truly new photos.